## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутстввовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

Постарайтесь провалидировать код с помощью assert-ов, вызываемых в ноутбуке или в .py-файлах.

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

In [19]:
import json

with open("laureates.json", encoding="utf-8") as file:
    laureates = json.load(file)


In [20]:
import importlib
import laureates_configs

importlib.reload(laureates_configs)
from laureates_configs import process_orgs, process_persons # реализуйте эти функции согласно описанной выше логике

def get_laureate_data(laureate):
    """
    Определите тип лауреата and соотв образом обработайте данных.
    
    Args:
        laureate: Dictionary with laureate data
        
    Returns:
        Processed laureate data with added 'type_' field ('person' or 'org')
        
    Determines type by:
    - 'knownName' key → person (uses process_persons)
    - 'orgName' key → org (uses process_orgs)
    """
    if "knownName" in laureate:
        result = process_persons(laureate)
        result["type_"] = "person"
    elif "orgName" in laureate:
        result = process_orgs(laureate)
        result["type_"] = "org"
    else:
        raise ValueError("Неизвестный тип лауреата")

    return result



laureates_data = []

for laureate in laureates['laureates']:
    laureates_data.append(get_laureate_data(laureate))

print("Обработано:", len(laureates_data))
print("Первая запись:")
print(laureates_data[0])

Обработано: 1028
Первая запись:
{'id': 745, 'name': 'A. Michael Spence', 'gender': 'male', 'birth_year': 1943, 'country_birth': 'USA', 'country_now': 'USA', 'prizes_relevant': [{'prize_amount': 10000000, 'prize_amount_adjusted': 15547541, 'award_year': 2001, 'category_en': 'Economic Sciences', 'prize_status': 'received'}], 'type_': 'person'}




### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

In [21]:

from pprint import pprint

from dataset_helpers import (
    count_missing,
    missing_share,
    find_duplicate_ids,
    find_id_gaps,
)


print("Всего записей:", len(laureates_data))

missing = count_missing(laureates_data)

print("\nВсе ли поля заполнены:", all(count == 0 for count in missing.values()))

print("\nКоличество пропусков:")
pprint(missing)

print("\nДоли пропусков:")
pprint(missing_share(laureates_data))

Всего записей: 1028

Все ли поля заполнены: False

Количество пропусков:
{'birth_year': 28,
 'country_birth': 31,
 'country_founded': 1004,
 'country_now': 7,
 'founded_year': 1001,
 'gender': 28,
 'id': 0,
 'name': 0,
 'prizes_relevant': 0,
 'type_': 0}

Доли пропусков:
{'birth_year': 0.0272,
 'country_birth': 0.0302,
 'country_founded': 0.9767,
 'country_now': 0.0068,
 'founded_year': 0.9737,
 'gender': 0.0272,
 'id': 0.0,
 'name': 0.0,
 'prizes_relevant': 0.0,
 'type_': 0.0}


In [22]:
# Разделяем людей и организации.
persons = []
organizations = []

for laureate in laureates_data:
    if laureate["type_"] == "person":
        persons.append(laureate)
    else:
        organizations.append(laureate)

print("Людей:", len(persons))
print("Организаций:", len(organizations))

print("\nПропуски у людей:")
pprint(count_missing(persons))

print("\nДоли пропусков у людей:")
pprint(missing_share(persons))

print("\nПропуски у организаций:")
pprint(count_missing(organizations))

print("\nДоли пропусков у организаций:")
pprint(missing_share(organizations))

Людей: 1000
Организаций: 28

Пропуски у людей:
{'birth_year': 0,
 'country_birth': 3,
 'country_now': 3,
 'gender': 0,
 'id': 0,
 'name': 0,
 'prizes_relevant': 0,
 'type_': 0}

Доли пропусков у людей:
{'birth_year': 0.0,
 'country_birth': 0.003,
 'country_now': 0.003,
 'gender': 0.0,
 'id': 0.0,
 'name': 0.0,
 'prizes_relevant': 0.0,
 'type_': 0.0}

Пропуски у организаций:
{'country_founded': 4,
 'country_now': 4,
 'founded_year': 1,
 'id': 0,
 'name': 0,
 'prizes_relevant': 0,
 'type_': 0}

Доли пропусков у организаций:
{'country_founded': 0.1429,
 'country_now': 0.1429,
 'founded_year': 0.0357,
 'id': 0.0,
 'name': 0.0,
 'prizes_relevant': 0.0,
 'type_': 0.0}


In [23]:
ids = []

for laureate in laureates_data:
    ids.append(laureate["id"])

min_id = min(ids)
max_id = max(ids)

print("Минимальный ID:", min_id)
print("Максимальный ID:", max_id)

# Показываем имена и годы премий для крайних ID.
for laureate in laureates_data:
    if laureate["id"] in (min_id, max_id):
        award_years = []

        for prize in laureate["prizes_relevant"]:
            award_years.append(prize["award_year"])

        print(
            "ID:", laureate["id"],
            "| Имя:", laureate["name"],
            "| Годы премий:", award_years,
        )

print("\nПовторяющиеся ID:")
pprint(find_duplicate_ids(laureates_data))

print("\nПары соседних ID с разрывом больше 1:")
pprint(find_id_gaps(laureates_data))

Минимальный ID: 1
Максимальный ID: 1070
ID: 1070 | Имя: Michael Woodford | Годы премий: [2026]
ID: 1 | Имя: Wilhelm Conrad Röntgen | Годы премий: [1901]

Повторяющиеся ID:
{}

Пары соседних ID с разрывом больше 1:
[(6, 8),
 (93, 95),
 (169, 172),
 (255, 257),
 (468, 470),
 (503, 505),
 (520, 523),
 (541, 543),
 (569, 571),
 (590, 592),
 (594, 596),
 (597, 600),
 (610, 613),
 (615, 617),
 (631, 633),
 (635, 637),
 (645, 647),
 (655, 657),
 (783, 786),
 (787, 789),
 (858, 861),
 (896, 903),
 (949, 958),
 (1009, 1011)]



Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

При анализе данных вам понадобятся более конкретные функции, например filter_by_num_prizes или output_metric_by_category. Их пишите прямо в ноутбуке и сопровождайте коментами.

Ниже мы приводим некоторые примеры вопросов, которые может быть интересно задать к данным, но если у вас есть желание, можете поисследовать датасет с любой интересной вам стороны!


**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

**2.4. Миграции лауреатов**
- У скольки лауреатов исходная страна и страна на момент вручения нобелевской премии различаются? Всегда ли это различие объясняется миграцией? Добавьте вспомогательное поле 'country_of_origin_adjusted_for_migration', чтобы различие между этим полем и страной на момент вручения объяснялось миграцией.
- Выявите страны с наибольшей долей "оттекших" и "притекших" лауреатов.
- Есть ли оттоки среди организаций?

**2.5. Многократные лауреаты**
- Найдите лауреатов, получившие больше одного приза. Кто получил больше 2 раз? 
- Совпадает ли категория второго приза с первой?  
- Сколько в среднем времени прошло между получением первой и второй премии?

**2.6 Отказы от премий**
- Определите, в каких категориях чаще отказывались от премий?
- Есть ли отказы среди организаций?

In [24]:
import importlib
import dataset_helpers

importlib.reload(dataset_helpers)

from dataset_helpers import filter_by_value, group_by_attributes
from aggregations import count_by_field, top_n


persons = filter_by_value(laureates_data, "type_", "person")

persons_with_country = []

for person in persons:
    if person["country_now"] is not None:
        persons_with_country.append(person)

print("Людей с известной страной:", len(persons_with_country))
print("Без страны:", len(persons) - len(persons_with_country))

country_counts = count_by_field(persons_with_country, "country_now")

print("\nТоп-5 стран рождения:")

for country in top_n(country_counts, "count"):
    print(country["country_now"], "—", country["count"])

Людей с известной страной: 997
Без страны: 3

Топ-5 стран рождения:
USA — 299
United Kingdom — 94
Germany — 86
France — 64
Japan — 31


In [25]:
persons_by_category = []

for person in persons_with_country:
    categories = []

    for prize in person["prizes_relevant"]:
        category = prize["category_en"]

        if category is not None and category not in categories:
            categories.append(category)

    for category in categories:
        persons_by_category.append({
            "country": person["country_now"],
            "category": category,
        })

category_groups = group_by_attributes(persons_by_category, ["category"])

for key, records in sorted(category_groups.items()):
    category = key[0]
    country_counts = count_by_field(records, "country")

    print("\nКатегория:", category)

    for country in top_n(country_counts, "count"):
        print(country["country"], "—", country["count"])


Категория: Chemistry
USA — 59
Germany — 26
United Kingdom — 26
France — 12
Japan — 9

Категория: Economic Sciences
USA — 56
United Kingdom — 7
Canada — 5
France — 5
the Netherlands — 4

Категория: Literature
France — 12
USA — 10
Sweden — 7
Poland — 7
Germany — 7

Категория: Peace
USA — 19
France — 10
Sweden — 5
Germany — 5
United Kingdom — 5

Категория: Physics
USA — 71
Germany — 27
United Kingdom — 25
France — 13
Japan — 12

Категория: Physiology or Medicine
USA — 85
United Kingdom — 25
Germany — 20
France — 12
Sweden — 8


In [26]:
persons_by_decade = []

for person in persons_with_country:
    award_years = []

    for prize in person["prizes_relevant"]:
        if prize["award_year"] is not None:
            award_years.append(prize["award_year"])

    if not award_years:
        continue

    first_year = min(award_years)
    decade = first_year // 10 * 10

    persons_by_decade.append({
        "country": person["country_now"],
        "decade": decade,
    })

decade_groups = group_by_attributes(persons_by_decade, ["decade"])

for key, records in sorted(decade_groups.items()):
    decade = key[0]
    country_counts = count_by_field(records, "country")

    print("\nДесятилетие:", decade)

    for country in top_n(country_counts, "count"):
        print(country["country"], "—", country["count"])


Десятилетие: 1900
France — 9
Germany — 8
Poland — 5
Switzerland — 4
Italy — 4

Десятилетие: 1910
Germany — 6
France — 6
Sweden — 3
USA — 3
the Netherlands — 3

Десятилетие: 1920
Germany — 10
France — 8
United Kingdom — 7
USA — 4
Denmark — 4

Десятилетие: 1930
USA — 14
United Kingdom — 10
Germany — 8
Austria — 4
France — 3

Десятилетие: 1940
USA — 13
United Kingdom — 3
Germany — 3
Czech Republic — 2
Denmark — 2

Десятилетие: 1950
USA — 21
United Kingdom — 10
Russia — 6
Germany — 6
France — 4

Десятилетие: 1960
USA — 20
United Kingdom — 7
France — 7
Germany — 7
Australia — 3

Десятилетие: 1970
USA — 32
United Kingdom — 13
Germany — 8
Russia — 6
Sweden — 6

Десятилетие: 1980
USA — 31
Germany — 11
Sweden — 5
France — 4
United Kingdom — 4

Десятилетие: 1990
USA — 42
Canada — 6
Poland — 6
United Kingdom — 6
Germany — 4

Десятилетие: 2000
USA — 52
United Kingdom — 10
Japan — 8
Germany — 6
France — 4

Десятилетие: 2010
USA — 39
Japan — 11
United Kingdom — 11
Canada — 5
France — 5

Десятилетие

In [29]:
import importlib
import aggregations

importlib.reload(aggregations)

from aggregations import calculate_statistics, top_n
from atribute_manager import add_attribute


def get_first_award_year(person):
    years = []

    for prize in person["prizes_relevant"]:
        if prize["award_year"] is not None:
            years.append(prize["award_year"])

    if not years:
        return None

    return min(years)


def get_first_award_age(person):
    birth_year = person["birth_year"]
    award_year = person["first_award_year"]

    if birth_year is None or award_year is None:
        return None

    return award_year - birth_year


def get_decade(person):
    year = person["first_award_year"]

    if year is None:
        return None

    return year // 10 * 10


persons_with_age = add_attribute(
    persons, "first_award_year", get_first_award_year
)

persons_with_age = add_attribute(
    persons_with_age, "age", get_first_award_age
)

persons_with_age = add_attribute(
    persons_with_age, "decade", get_decade
)

In [30]:
age_data = []

for person in persons_with_age:
    if person["age"] is not None:
        age_data.append(person)

print("Людей с известным возрастом:", len(age_data))
print("Без возраста:", len(persons_with_age) - len(age_data))

statistics = calculate_statistics(age_data, "age")

print("\nСредний возраст:", statistics["mean"])
print("Медианный возраст:", statistics["median"])
print("Максимальный возраст:", statistics["max"])

print("\nТоп-5 самых старших при получении первой премии:")

for person in top_n(age_data, "age"):
    print(
        person["name"],
        "| Возраст:", person["age"],
        "| Год:", person["first_award_year"],
    )

Людей с известным возрастом: 1000
Без возраста: 0

Средний возраст: 60.61
Медианный возраст: 61.0
Максимальный возраст: 97

Топ-5 самых старших при получении первой премии:
John B. Goodenough | Возраст: 97 | Год: 2019
Arthur Ashkin | Возраст: 96 | Год: 2018
Henri B. Kagan | Возраст: 96 | Год: 2026
John J. Hopfield | Возраст: 91 | Год: 2024
Klaus Hasselmann | Возраст: 90 | Год: 2021


In [31]:
ages_by_category = []

for person in age_data:
    categories = []

    for prize in person["prizes_relevant"]:
        if prize["award_year"] == person["first_award_year"]:
            category = prize["category_en"]

            if category is not None and category not in categories:
                categories.append(category)

    for category in categories:
        ages_by_category.append({
            "category": category,
            "age": person["age"],
        })

groups = group_by_attributes(ages_by_category, ["category"])

for key, records in sorted(groups.items()):
    statistics = calculate_statistics(records, "age")

    print(
        key[0],
        "| Людей:", statistics["count"],
        "| Среднее:", statistics["mean"],
        "| Медиана:", statistics["median"],
        "| Максимум:", statistics["max"],
    )

Chemistry | Людей: 199 | Среднее: 59.43 | Медиана: 58 | Максимум: 97
Economic Sciences | Людей: 101 | Среднее: 67.21 | Медиана: 67 | Максимум: 90
Literature | Людей: 123 | Среднее: 65.04 | Медиана: 67 | Максимум: 88
Peace | Людей: 112 | Среднее: 60.97 | Медиана: 62.0 | Максимум: 87
Physics | Людей: 230 | Среднее: 57.79 | Медиана: 56.0 | Максимум: 96
Physiology or Medicine | Людей: 235 | Среднее: 59.02 | Медиана: 58 | Максимум: 87


In [32]:
groups = group_by_attributes(age_data, ["decade"])

for key, records in sorted(groups.items()):
    statistics = calculate_statistics(records, "age")

    print(
        key[0],
        "| Людей:", statistics["count"],
        "| Среднее:", statistics["mean"],
        "| Медиана:", statistics["median"],
        "| Максимум:", statistics["max"],
    )

1900 | Людей: 56 | Среднее: 57.84 | Медиана: 58.0 | Максимум: 85
1910 | Людей: 37 | Среднее: 53.08 | Медиана: 50 | Максимум: 80
1920 | Людей: 54 | Среднее: 54.98 | Медиана: 56.0 | Максимум: 86
1930 | Людей: 55 | Среднее: 51.62 | Медиана: 49 | Максимум: 75
1940 | Людей: 40 | Среднее: 58.02 | Медиана: 56.0 | Максимум: 81
1950 | Людей: 71 | Среднее: 54.32 | Медиана: 54 | Максимум: 79
1960 | Людей: 74 | Среднее: 56.45 | Медиана: 57.0 | Максимум: 87
1970 | Людей: 102 | Среднее: 59.34 | Медиана: 59.0 | Максимум: 87
1980 | Людей: 93 | Среднее: 60.54 | Медиана: 60 | Максимум: 83
1990 | Людей: 101 | Среднее: 62.14 | Медиана: 62 | Максимум: 87
2000 | Людей: 119 | Среднее: 65.87 | Медиана: 64 | Максимум: 90
2010 | Людей: 117 | Среднее: 66.85 | Медиана: 69 | Максимум: 97
2020 | Людей: 81 | Среднее: 69.38 | Медиана: 69 | Максимум: 96


In [33]:
importlib.reload(aggregations)

from aggregations import count_share


statistics = count_share(persons_with_age, "gender", "female")

print("Всего людей:", len(persons_with_age))
print("Женщин:", statistics["count"])
print("Доля женщин, %:", statistics["percent"])
print("Не указан пол:", statistics["missing"])

Всего людей: 1000
Женщин: 69
Доля женщин, %: 6.9
Не указан пол: 0


In [34]:
persons_by_category = []

for person in persons_with_age:
    categories = []

    for prize in person["prizes_relevant"]:
        category = prize["category_en"]

        if category is not None and category not in categories:
            categories.append(category)

    for category in categories:
        persons_by_category.append({
            "category": category,
            "gender": person["gender"],
        })

groups = group_by_attributes(persons_by_category, ["category"])

for key, records in sorted(groups.items()):
    statistics = count_share(records, "gender", "female")

    print(
        key[0],
        "| Всего людей:", len(records),
        "| Женщин:", statistics["count"],
        "| Доля женщин, %:", statistics["percent"],
    )

Chemistry | Всего людей: 200 | Женщин: 8 | Доля женщин, %: 4.0
Economic Sciences | Всего людей: 101 | Женщин: 3 | Доля женщин, %: 2.97
Literature | Всего людей: 123 | Женщин: 19 | Доля женщин, %: 15.45
Peace | Всего людей: 113 | Женщин: 21 | Доля женщин, %: 18.58
Physics | Всего людей: 230 | Женщин: 5 | Доля женщин, %: 2.17
Physiology or Medicine | Всего людей: 235 | Женщин: 14 | Доля женщин, %: 5.96


In [35]:
persons_with_decade = []

for person in persons_with_age:
    if person["decade"] is not None:
        persons_with_decade.append(person)

print(
    "Без года первой премии:",
    len(persons_with_age) - len(persons_with_decade),
)

groups = group_by_attributes(persons_with_decade, ["decade"])

for key, records in sorted(groups.items()):
    statistics = count_share(records, "gender", "female")

    print(
        key[0],
        "| Всего людей:", len(records),
        "| Женщин:", statistics["count"],
        "| Доля женщин, %:", statistics["percent"],
    )

Без года первой премии: 0
1900 | Всего людей: 56 | Женщин: 3 | Доля женщин, %: 5.36
1910 | Всего людей: 37 | Женщин: 0 | Доля женщин, %: 0.0
1920 | Всего людей: 54 | Женщин: 2 | Доля женщин, %: 3.7
1930 | Всего людей: 55 | Женщин: 3 | Доля женщин, %: 5.45
1940 | Всего людей: 40 | Женщин: 3 | Доля женщин, %: 7.5
1950 | Всего людей: 71 | Женщин: 0 | Доля женщин, %: 0.0
1960 | Всего людей: 74 | Женщин: 3 | Доля женщин, %: 4.05
1970 | Всего людей: 102 | Женщин: 4 | Доля женщин, %: 3.92
1980 | Всего людей: 93 | Женщин: 4 | Доля женщин, %: 4.3
1990 | Всего людей: 101 | Женщин: 7 | Доля женщин, %: 6.93
2000 | Всего людей: 119 | Женщин: 11 | Доля женщин, %: 9.24
2010 | Всего людей: 117 | Женщин: 13 | Доля женщин, %: 11.11
2020 | Всего людей: 81 | Женщин: 16 | Доля женщин, %: 19.75


In [36]:
persons_by_category_decade = []

for person in persons_with_age:
    counted_pairs = []

    for prize in person["prizes_relevant"]:
        category = prize["category_en"]
        year = prize["award_year"]

        if category is None or year is None:
            continue

        decade = year // 10 * 10
        pair = (category, decade)

        if pair not in counted_pairs:
            counted_pairs.append(pair)

            persons_by_category_decade.append({
                "category": category,
                "decade": decade,
                "gender": person["gender"],
            })

groups = group_by_attributes(
    persons_by_category_decade, ["decade", "category"]
)

for key, records in sorted(groups.items()):
    decade, category = key
    statistics = count_share(records, "gender", "female")

    print(
        decade,
        "|", category,
        "| Женщин:", statistics["count"],
        "| Доля женщин, %:", statistics["percent"],
    )

1900 | Chemistry | Женщин: 0 | Доля женщин, %: 0.0
1900 | Literature | Женщин: 1 | Доля женщин, %: 10.0
1900 | Peace | Женщин: 1 | Доля женщин, %: 7.69
1900 | Physics | Женщин: 1 | Доля женщин, %: 7.69
1900 | Physiology or Medicine | Женщин: 0 | Доля женщин, %: 0.0
1910 | Chemistry | Женщин: 1 | Доля женщин, %: 12.5
1910 | Literature | Женщин: 0 | Доля женщин, %: 0.0
1910 | Peace | Женщин: 0 | Доля женщин, %: 0.0
1910 | Physics | Женщин: 0 | Доля женщин, %: 0.0
1910 | Physiology or Medicine | Женщин: 0 | Доля женщин, %: 0.0
1920 | Chemistry | Женщин: 0 | Доля женщин, %: 0.0
1920 | Literature | Женщин: 2 | Доля женщин, %: 20.0
1920 | Peace | Женщин: 0 | Доля женщин, %: 0.0
1920 | Physics | Женщин: 0 | Доля женщин, %: 0.0
1920 | Physiology or Medicine | Женщин: 0 | Доля женщин, %: 0.0
1930 | Chemistry | Женщин: 1 | Доля женщин, %: 7.69
1930 | Literature | Женщин: 1 | Доля женщин, %: 11.11
1930 | Peace | Женщин: 1 | Доля женщин, %: 12.5
1930 | Physics | Женщин: 0 | Доля женщин, %: 0.0
193

## Главное
## Основные выводы

### Страны рождения
- **США лидируют по числу родившихся там лауреатов:** 299 из 997 людей с известной страной рождения — около 30%. Далее идут Великобритания (94), Германия (86), Франция (64) и Япония (31).
- Лидер зависит от категории: в химии и экономике первое место занимают США — 59 и 56 человек соответственно. В литературе лидирует Франция — 12 человек.
- География менялась со временем: среди впервые награждённых в 1900-х лидировала Франция, в 1920-х — Германия, а в 1930-х — США.

### Возраст первой премии
- **Средний возраст получения первой премии — 60,61 года, медиана — 61 год.** Эти показатели близки. Максимальный возраст по разнице годов — 97 лет у John B. Goodenough.
- Самый высокий средний возраст первой премии наблюдается в экономике — **67,21 года**, самый низкий — в физике: **57,79 года**. Разница составляет около 9,4 года.
- С 1950-х средний возраст первой премии последовательно увеличивался: с **54,32 года** до **66,85 года в 2010-х** и **69,38 года в доступной части 2020-х**. Это показывает изменение возраста награждения, но не объясняет его причины.

### Гендерное распределение
- Среди 1000 людей-лауреатов **69 женщин — 6,9%**. В целом за исследуемый период женщины составляют небольшую часть лауреатов.
- Представленность женщин различается по категориям. Наибольшая доля — среди лауреатов премии мира (**18,58%**) и по литературе (**15,45%**), наименьшая — по физике (**2,17%**) и экономике (**2,97%**).
- **Доля женщин среди впервые награждённых растёт в последние десятилетия:** 6,93% в 1990-х, 9,24% в 2000-х, 11,11% в 2010-х и 19,75% в доступной части 2020-х. Поэтому общий показатель за всю историю скрывает заметные изменения последних лет.



---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций**

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-IDF + сравнените топ слов по этой метрике с просто самыми популярными словами)